# Policy concept classification

This Snowflake Notebook extracts atomic policy ideas from the AI impact survey, compares them with the existing policy taxonomy, proposes and validates candidate additions, and classifies every extracted idea against a frozen taxonomy.

The notebook uses the active Snowpark session and `SNOWFLAKE.CORTEX.COMPLETE`; it does not establish a separate Snowflake connection. Intermediate results are displayed in notebook cells and remain in memory. Optional write cells at the end are disabled by default and apply only to the frozen taxonomy and final classifications.

Run cells from top to bottom. Start with `REVIEW_MODE = True` and the lower-cost model to inspect the taxonomy and QA outputs. Set `REVIEW_MODE = False` only after reviewing the validation result. Set `ENABLE_OUTPUT_WRITES = True` only when you explicitly want to persist the two final outputs.

In [ ]:
# Import libraries used for hashing, JSON handling, timestamps, SQL generation, and dataframe display.
import hashlib
import json
import os
import re
import time
import uuid
from datetime import datetime, timezone

import pandas as pd
from dotenv import load_dotenv
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas

# Load local environment variables for Snowflake credentials and Cortex model names.
load_dotenv()

# Use the production dbt source objects currently configured for this analysis.
SOURCE_DATABASE = "TRANSFORM_ENGCA_PRD"
SOURCE_GOVOCAL_SCHEMA = "GOVOCAL"
SOURCE_AI_SCHEMA = "AI_ENGAGEMENT"
SURVEY_TABLE = "INT_GOVOCAL_AI_SURVEY"
TAXONOMY_TABLE = "STG_PHASE2_POLICY_CONCEPTS_AND_THEMES"

# Select a lower-cost model for iteration and a higher-cost model for final runs.
CORTEX_MODEL = os.environ.get("LLM_MODEL_HIGH", "")
ITERATION_MODEL = os.environ.get("LLM_MODEL_LOW", "")
USE_ITERATION_MODEL = True
REVIEW_MODE = False  # True displays the taxonomy review and skips final classification.

# Record metadata for the current in-memory run and retry transient failures.
PROMPT_VERSION = "policy-concepts-v3-set-based-sql-local"
RUN_ID = str(uuid.uuid4())
RUN_TIMESTAMP = datetime.now(timezone.utc).isoformat()
MAX_RETRIES = int(os.environ.get("POLICY_CLASSIFICATION_MAX_RETRIES", "3"))

# Use the lower-cost model for iteration and require an explicit model configuration.
if USE_ITERATION_MODEL:
    CORTEX_MODEL = ITERATION_MODEL
if not CORTEX_MODEL:
    raise ValueError("Set LLM_MODEL_LOW or LLM_MODEL_HIGH in the local environment.")

# Keep only government-action extraction active; add the economic field here when ready.
EXTRACTION_FIELDS = ["government_action_suggestion"]

# Preserve optional final writes, disabled by default.
ENABLE_OUTPUT_WRITES = False
TARGET_DATABASE = os.environ.get("POLICY_CLASSIFICATION_DATABASE", "TRANSFORM_ENGCA_DEV")
TARGET_SCHEMA = os.environ.get("POLICY_CLASSIFICATION_SCHEMA", os.environ.get("SNOWFLAKE_SCHEMA", "DBT_CHOLLINGSWORTH_AI_ENGAGEMENT"))
OPTIONAL_TAXONOMY_TABLE = "POLICY_TAXONOMY_FROZEN"
OPTIONAL_CLASSIFICATION_TABLE = "POLICY_IDEA_CLASSIFICATIONS"

# Keep this connection open so cells can be run sequentially in VS Code.
conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    authenticator=os.environ.get("SNOWFLAKE_AUTHENTICATOR", "externalbrowser"),
    role=os.environ.get("SNOWFLAKE_ROLE", ""),
    warehouse=os.environ.get("SNOWFLAKE_WAREHOUSE", ""),
    database=TARGET_DATABASE,
    schema=TARGET_SCHEMA,
)
cur = conn.cursor()

print(f"run_id={RUN_ID} model={CORTEX_MODEL} review_mode={REVIEW_MODE}")

/home/ch/caldata-engaged-california/.venv/lib/python3.13/site-packages/snowflake/connector/vendored/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (7.6.0)/charset_normalizer (3.5.1) doesn't match a supported version!
  warnings.warn(


Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://login.innovation.ca.gov/app/snowflake/exkc5fv0t9YwtoE2p697/sso/saml?SAMLRequest=jVJdc9owEPwrHvUZSTYfwRog40CYkCYNA0476ZuwBajYktHJGPrrKxvopA%2FJ9E1z2r3du73B7THPvIMwILUaIh9T5AmV6FSqzRC9xtNWH3lguUp5ppUYopMAdDsaAM%2BzgkWl3aqF2JcCrOcaKWD1xxCVRjHNQQJTPBfAbMKW0fMTCzBlHEAY6%2BTQhZKCdFpbawtGSFVVuGpjbTYkoJQSGhKHqiFf0DuJ4nONwmirE51dKUc30wcSPqGdWsIhnML8QryT6ryCz1RWZxCwhziet%2BYvyxh50XW6sVZQ5sIshTnIRLwuns4GwDnYilXHp2EXg9LVOuM7kei8KK1rht2LrEVKMr2RbkWzyRAVO5nu9%2BU%2BLE9Sv21%2F3M27v1%2BgeohWi1%2Fd53H1OP3KV3HUV%2F32dvV4nyDv%2BzXQoA50BlCKmapjtK5Eg16Lhi3aiWmHtX0WhLh3E%2F5E3sTFKBW3DfPqtfGBpVL60PzghOONPhBeFOSvfSKOu6S7PlAbvlVW3wdFL7whAJrUYaHzvbDGhxn95xYG5D3pcnHfXAizyVxnMjl5U21ybj%2FOyMd%2BU5Fpa91Amci5zKI0NQLAZZVluhobwa07bGtKgcjorPrvaY%2F%2BAA%3D%3D&RelayState=ver%3A3-hint%3A8769185083035654-ETMsDgAAAaBqr5%2BPABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAEJEI6IzxAHRWYVOvOciZdqQAAACgx6BwgrbtpePqI%2BBwetsm

/usr/bin/xdg-open: 882: x-www-browser: not found
/usr/bin/xdg-open: 882: firefox: not found
/usr/bin/xdg-open: 882: iceweasel: not found
/usr/bin/xdg-open: 882: seamonkey: not found
/usr/bin/xdg-open: 882: mozilla: not found
/usr/bin/xdg-open: 882: epiphany: not found
/usr/bin/xdg-open: 882: konqueror: not found
/usr/bin/xdg-open: 882: chromium: not found
/usr/bin/xdg-open: 882: chromium-browser: not found
/usr/bin/xdg-open: 882: google-chrome: not found
/usr/bin/xdg-open: 882: www-browser: not found
/usr/bin/xdg-open: 882: links2: not found
/usr/bin/xdg-open: 882: elinks: not found
/usr/bin/xdg-open: 882: links: not found
/usr/bin/xdg-open: 882: lynx: not found
/usr/bin/xdg-open: 882: w3m: not found
xdg-open: no method available for opening 'https://login.innovation.ca.gov/app/snowflake/exkc5fv0t9YwtoE2p697/sso/saml?SAMLRequest=jVJdc9owEPwrHvUZSTYfwRog40CYkCYNA0476ZuwBajYktHJGPrrKxvopA%2FJ9E1z2r3du73B7THPvIMwILUaIh9T5AmV6FSqzRC9xtNWH3lguUp5ppUYopMAdDsaAM%2BzgkWl3aqF2JcCrOcaKWD1xxCVRjH

run_id=d433ef6f-2b30-47f1-a3e9-c03c40618128 model=openai-gpt-5-mini review_mode=True


In [2]:
def qident(name: str) -> str:
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_$]*", name):
        raise ValueError(f"Unsafe identifier: {name}")
    return f'"{name.upper()}"'


def fq_table(database: str, schema: str, table: str) -> str:
    return ".".join(qident(part) for part in (database, schema, table))


conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    authenticator=os.environ.get("SNOWFLAKE_AUTHENTICATOR", "externalbrowser"),
    role=os.environ.get("SNOWFLAKE_ROLE", ""),
    warehouse=os.environ.get("SNOWFLAKE_WAREHOUSE", ""),
    database=TARGET_DATABASE,
    schema=TARGET_SCHEMA,
)

SOURCE_SURVEY = fq_table(SOURCE_DATABASE, SOURCE_GOVOCAL_SCHEMA, SURVEY_TABLE)
SOURCE_TAXONOMY = fq_table(SOURCE_DATABASE, SOURCE_AI_SCHEMA, TAXONOMY_TABLE)
TARGET_PREFIX = lambda table: fq_table(TARGET_DATABASE, TARGET_SCHEMA, table)


def query_df(sql: str, params: list | None = None) -> pd.DataFrame:
    cur.execute(sql, params or [])
    return cur.fetch_pandas_all()


def sql_literal(value: object) -> str:
    if value is None or pd.isna(value):
        return "NULL"
    return "'" + str(value).replace("'", "''") + "'"


def values_cte(frame: pd.DataFrame, columns: list[str], cte_name: str = "input_rows") -> str:
    if frame.empty:
        return f"{cte_name} ({', '.join(columns)}) AS (SELECT * FROM VALUES (NULL) WHERE FALSE)"
    rows = ",\n".join(
        "(" + ", ".join(sql_literal(row[column]) for column in columns) + ")"
        for _, row in frame[columns].iterrows()
    )
    aliases = ", ".join(columns)
    return f"{cte_name} ({aliases}) AS (SELECT * FROM VALUES\n{rows})"


def extract_structured_response(raw: str) -> str:
    payload = json.loads(raw)
    structured = payload.get("structured_output")
    if isinstance(structured, list) and structured:
        first = structured[0]
        raw_message = first.get("raw_message")
        if isinstance(raw_message, dict):
            return json.dumps(raw_message)
        if isinstance(raw_message, str):
            return raw_message
    choices = payload.get("choices")
    if isinstance(choices, list) and choices:
        first_choice = choices[0]
        messages = first_choice.get("messages")
        if isinstance(messages, str):
            return messages
        if isinstance(messages, list):
            content = "".join(msg.get("content", "") for msg in messages if isinstance(msg, dict))
            if content:
                return content
    raise ValueError("Unexpected Snowflake Cortex response shape")


def table_exists(table: str) -> bool:
    try:
        conn.cursor().execute(f"SELECT 1 FROM {table} LIMIT 1")
        return True
    except Exception:
        return False


def write_output(frame: pd.DataFrame, table_name: str, key_columns: list[str] | None = None) -> None:
    if frame.empty:
        return
    table = TARGET_PREFIX(table_name)
    if key_columns and table_exists(table):
        key_frame = frame[key_columns].drop_duplicates()
        for _, key in key_frame.iterrows():
            predicates = " AND ".join(f"{qident(column)} = %s" for column in key_columns)
            conn.cursor().execute(
                f"DELETE FROM {table} WHERE {predicates}",
                [None if pd.isna(key[column]) else key[column] for column in key_columns],
            )
    write_pandas(
        conn,
        frame.reset_index(drop=True),
        table_name.upper(),
        database=TARGET_DATABASE,
        schema=TARGET_SCHEMA,
        auto_create_table=True,
        overwrite=False,
        quote_identifiers=True,
    )


def cortex_json(messages: list[dict], response_schema: dict, max_tokens: int = 1200) -> tuple[dict | None, str, str | None]:
    payload = json.dumps(messages)
    schema = json.dumps({"type": "json", "schema": response_schema})
    last_error = None
    for attempt in range(MAX_RETRIES):
        try:
            cursor = conn.cursor()
            cursor.execute(
                """SELECT SNOWFLAKE.CORTEX.COMPLETE(
                    %s, PARSE_JSON(%s),
                    OBJECT_CONSTRUCT('temperature', 0, 'max_tokens', %s,
                                     'response_format', PARSE_JSON(%s))
                ) AS RESULT""",
                [CORTEX_MODEL, payload, max_tokens, schema],
            )
            raw = cursor.fetchone()[0]
            content = extract_structured_response(raw)
            content = re.sub(r"^```(?:json)?\s*", "", content.strip()).rstrip("` \n")
            return json.loads(content), raw, None
        except Exception as exc:
            last_error = f"{type(exc).__name__}: {exc}"
            if attempt + 1 < MAX_RETRIES:
                time.sleep(2 ** attempt)
    return None, "", last_error


def stable_id(*parts: str) -> str:
    return hashlib.md5("|".join(str(part).strip() for part in parts).encode("utf-8")).hexdigest()


def text_value(value) -> str:
    return "" if pd.isna(value) else str(value).strip()

print(SOURCE_SURVEY)
print(SOURCE_TAXONOMY)


Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://login.innovation.ca.gov/app/snowflake/exkc5fv0t9YwtoE2p697/sso/saml?SAMLRequest=jVJdc9owEPwrHvUZSXaAYA2QIXxM6SSBBtJpeVNsARpsSdXJNvz7ygYyyUMyfdOcdm%2F3bq9%2Fd8yzoBQWpFYDFGKKAqESnUq1G6CX9azVQwE4rlKeaSUG6CQA3Q37wPPMsFHh9upZ%2FC0EuMA3UsDqjwEqrGKagwSmeC6AuYStRo8PLMKUcQBhnZdDF0oK0mvtnTOMkKqqcHWDtd2RiFJKaEw8qoZ8Q%2B8kzNcaxmqnE51dKUc%2F0ycSIaHtWsIjvMLyQryX6ryCr1RezyBg39frZWu5WK1RMLpON9YKilzYlbClTMTL88PZAHgHe%2FHaDmncwaB0tc34QSQ6N4XzzbB%2Fka1ISaZ30q9oPhkgc5DptuAbs%2Bw8ju6jdMNnMhkfaVSE04laTMXvH9V%2B0T2ponQm%2Fpmg4Nc10KgOdA5QiLmqY3S%2BRKNui8Yt2l7TNrsJWaeHb3vtDQomPkapuGuYV6%2BNDyyV0mXzgxOOd7ok3BjyZp%2BI4yHpbEvq4j%2BV09PIdONbAqBJHRY63wtrfNjhf26hT96TLhf35EOYT5Y6k8kpmGmbc%2Fd5RiEOm4pMW9sGykTOZTZKUysAfFZZpquxFdz5w3a2EIgMz6ofT3v4Dw%3D%3D&RelayState=ver%3A3-hint%3A8769185083035654-ETMsDgAAAaBqsBFAABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAEHwea9lfS8FAnqHW2tiZTQYAAACgxABLdcX1nm3UPbT124GV%2B00qkrp7

/usr/bin/xdg-open: 882: x-www-browser: not found
/usr/bin/xdg-open: 882: firefox: not found
/usr/bin/xdg-open: 882: iceweasel: not found
/usr/bin/xdg-open: 882: seamonkey: not found
/usr/bin/xdg-open: 882: mozilla: not found
/usr/bin/xdg-open: 882: epiphany: not found
/usr/bin/xdg-open: 882: konqueror: not found
/usr/bin/xdg-open: 882: chromium: not found
/usr/bin/xdg-open: 882: chromium-browser: not found
/usr/bin/xdg-open: 882: google-chrome: not found
/usr/bin/xdg-open: 882: www-browser: not found
/usr/bin/xdg-open: 882: links2: not found
/usr/bin/xdg-open: 882: elinks: not found
/usr/bin/xdg-open: 882: links: not found
/usr/bin/xdg-open: 882: lynx: not found
/usr/bin/xdg-open: 882: w3m: not found
xdg-open: no method available for opening 'https://login.innovation.ca.gov/app/snowflake/exkc5fv0t9YwtoE2p697/sso/saml?SAMLRequest=jVJdc9owEPwrHvUZSXaAYA2QIXxM6SSBBtJpeVNsARpsSdXJNvz7ygYyyUMyfdOcdm%2F3bq9%2Fd8yzoBQWpFYDFGKKAqESnUq1G6CX9azVQwE4rlKeaSUG6CQA3Q37wPPMsFHh9upZ%2FC0EuMA3UsDqjwEqr

"TRANSFORM_ENGCA_PRD"."GOVOCAL"."INT_GOVOCAL_AI_SURVEY"
"TRANSFORM_ENGCA_PRD"."AI_ENGAGEMENT"."STG_PHASE2_POLICY_CONCEPTS_AND_THEMES"


In [18]:
# Query published survey responses that contain at least one policy-relevant answer.
survey_sql = f"""
SELECT  TOP 20 -- top 20 is temporary for testing
    survey_id,
    government_action_suggestion,
    economic_impact_expectation
FROM {SOURCE_SURVEY}
WHERE LOWER(COALESCE(publication_status, 'published')) = 'published'
  AND (
      NULLIF(TRIM(government_action_suggestion), '') IS NOT NULL
      OR NULLIF(TRIM(economic_impact_expectation), '') IS NOT NULL
  )
"""

# Query the existing taxonomy and add empty criteria columns for the validation stage.
taxonomy_sql = f"SELECT policy_concept_id, policy_concept, policy_concept_description, subtheme, theme FROM {SOURCE_TAXONOMY}"
survey_df = query_df(survey_sql)
taxonomy_df = query_df(taxonomy_sql).fillna("")
taxonomy_df["CONCEPT_STATUS"] = "existing"
taxonomy_df["INCLUSION_CRITERIA"] = ""
taxonomy_df["EXCLUSION_CRITERIA"] = ""

# Confirm that both source objects contain the fields required by the pipeline.
assert {"SURVEY_ID", "GOVERNMENT_ACTION_SUGGESTION", "ECONOMIC_IMPACT_EXPECTATION"}.issubset(survey_df.columns)
assert {"POLICY_CONCEPT_ID", "POLICY_CONCEPT", "POLICY_CONCEPT_DESCRIPTION"}.issubset(taxonomy_df.columns)

# Display source counts as the first visible data checkpoint.
print(f"survey responses with relevant text: {len(survey_df):,}")
print(f"existing taxonomy concepts: {len(taxonomy_df):,}")
survey_df

survey responses with relevant text: 20
existing taxonomy concepts: 27


,SURVEY_ID,GOVERNMENT_ACTION_SUGGESTION,ECONOMIC_IMPACT_EXPECTATION
0,c033a551-79e4-48f8-b7e8-dd13a4d74614,"Oversight for safety, but don't let it become ...",Yes
1,75be846d-db36-4cc1-a854-3e979fd81c23,Limit AI use since it harms peoples. Look at o...,Unnecessary bloat and costs. Increase of AI jo...
2,9b185225-611f-426a-81de-72a35ec4fe22,I believe that AI should be heavily regulated....,I don't think I need to explain what happens t...
3,e22fcea4-7947-45f2-890b-71118fba2e01,Do not allow AI development companies to hide ...,I don't really care how it does - as far as I ...
4,dd80c22f-9f86-494a-8fa6-80de036d7673,Curb AI usage immediately and close data cente...,Negatively; if no one has money because they h...
5,5ad59128-47c2-41c8-9843-6b64fff639d2,"Ban AI data-centers, especially in agricultura...","I cannot speak for all areas, but I expect it ..."
6,16f953d3-998f-4142-8820-1f416e209cbd,The government needs to severely limit the use...,None
7,7fcf5792-09d9-4735-bb25-2517a4e40980,"I think the government should reign in AI, cre...",I expect AI to crash the economy soon due to s...
8,21253a90-0d19-4fed-becb-c11c6f0e419f,Enact limits similar to WARN notifications for...,more companies will continue to reduce headcou...
9,45e3dc2f-3a41-4699-9c14-3896ddb2fa6a,AI must be seen as a product of all humanity a...,I expect it will force us to change how we com...


In [15]:
# Define the prompt for extracting explicit atomic policy recommendations.
EXTRACTION_PROMPT = """
Extract all distinct, explicit policy recommendations from the single survey response below.
An atomic idea is one government, firm, school, or institutional action that could independently be accepted or rejected.
Split independent recommendations. Do not infer a recommendation from a condition, prediction, complaint, or consequence.
For example, saying regulation could increase employment is not a recommendation to regulate unless regulation is explicitly advocated.
Preserve the respondent's wording as much as possible and return an empty list when no explicit recommendation appears.
Return JSON only with this shape:
{"ideas": [{"idea_text": "...", "extraction_rationale": "..."}]}
""".strip()

# Define the prompt for provisional classification against the existing taxonomy.
BASELINE_PROMPT = """
Classify this atomic policy idea against the existing taxonomy below. Use descriptions, themes, and subthemes.
Do not force a fit. Use status classified only when exactly one concept is the best fit; use ambiguous when two or more are equally plausible; use no_fit when none adequately represents the idea.
Return JSON only with this shape:
{"classification_status":"classified|ambiguous|no_fit", "policy_concept_ids":[], "confidence":"high|medium|low", "rationale":"..."}
For classified, policy_concept_ids must contain exactly one existing ID. For ambiguous, include zero or more plausible existing IDs. For no_fit, it must be empty.
""".strip()

# Define the prompt for proposing candidate concepts for taxonomy gaps.
GAP_PROMPT = """
Review the no_fit and ambiguous atomic policy ideas against the existing taxonomy. Propose candidate policy concepts only for real gaps.
Policy concepts should be specific policy actions or instruments at a similar granularity to existing concepts, not broad objectives or duplicate labels.
Return JSON only with this shape:
{"candidates":[{"candidate_name":"...","definition":"...","supporting_idea_ids":[],"why_existing_insufficient":"...","closest_existing_concepts":[],"boundary":"..."}]}
""".strip()

# Define the prompt for checking taxonomy coverage, boundaries, and granularity.
VALIDATION_PROMPT = """
Validate the complete existing-plus-candidate taxonomy for classifying the supplied atomic policy ideas.
Check coverage, overlap, duplicates, parent/child subset relationships, mixed dimensions, multi-bucket ambiguity, granularity, and operational distinguishability.
Do not rewrite or remove existing concepts. Give a boundary/decision rule for each material overlap and identify concepts needing review.
Return JSON only with this shape:
{"is_ready":true,"findings":[{"severity":"high|medium|low","finding_type":"overlap|duplicate|parent_child|mixed_dimension|coverage|granularity|ambiguity|other","concept_ids":[],"detail":"...","decision_rule":"..."}]}
""".strip()

# Define the prompt for final classification against the frozen taxonomy.
FINAL_PROMPT = """
Classify one atomic policy idea against the frozen taxonomy. Do not create, rename, merge, split, or reinterpret concepts.
Return JSON only with this shape:
{"classification_status":"classified|ambiguous|no_fit", "policy_concept_id":null, "confidence":"high|medium|low", "rationale":"..."}
Use exactly one policy_concept_id only for classified. Use null for ambiguous or no_fit.
""".strip()

# Provide response schemas used to request and validate structured Cortex output.
def object_schema(properties: dict, required: list[str]) -> dict:
    return {
        "type": "object",
        "properties": properties,
        "required": required,
        "additionalProperties": False,
    }


cortex_schema = object_schema


EXTRACTION_SCHEMA = object_schema(
    {
        "ideas": {
            "type": "array",
            "items": object_schema(
                {
                    "idea_text": {"type": "string"},
                    "extraction_rationale": {"type": "string"},
                },
                ["idea_text", "extraction_rationale"],
            ),
        }
    },
    ["ideas"],
)
CLASSIFICATION_SCHEMA = object_schema(
    {
        "classification_status": {"type": "string"},
        "policy_concept_ids": {"type": "array", "items": {"type": "string"}},
        "confidence": {"type": "string"},
        "rationale": {"type": "string"},
    },
    ["classification_status", "policy_concept_ids", "confidence", "rationale"],
)
FINAL_SCHEMA = object_schema(
    {
        "classification_status": {"type": "string"},
        "policy_concept_id": {"type": ["string", "null"]},
        "confidence": {"type": "string"},
        "rationale": {"type": "string"},
    },
    ["classification_status", "policy_concept_id", "confidence", "rationale"],
)
GAP_SCHEMA = object_schema(
    {
        "candidates": {
            "type": "array",
            "items": object_schema(
                {
                    "candidate_name": {"type": "string"},
                    "definition": {"type": "string"},
                    "supporting_idea_ids": {"type": "array", "items": {"type": "string"}},
                    "why_existing_insufficient": {"type": "string"},
                    "closest_existing_concepts": {"type": "array", "items": {"type": "string"}},
                    "boundary": {"type": "string"},
                },
                ["candidate_name", "definition", "supporting_idea_ids", "why_existing_insufficient", "closest_existing_concepts", "boundary"],
            ),
        }
    },
    ["candidates"],
)
VALIDATION_SCHEMA = object_schema(
    {
        "is_ready": {"type": "boolean"},
        "findings": {
            "type": "array",
            "items": object_schema(
                {
                    "severity": {"type": "string"},
                    "finding_type": {"type": "string"},
                    "concept_ids": {"type": "array", "items": {"type": "string"}},
                    "detail": {"type": "string"},
                    "decision_rule": {"type": "string"},
                },
                ["severity", "finding_type", "concept_ids", "detail", "decision_rule"],
            ),
        },
    },
    ["is_ready", "findings"],
)


# Validate extracted ideas before they are persisted.
def validate_extraction(payload: dict | None) -> list[dict]:
    if not isinstance(payload, dict) or not isinstance(payload.get("ideas"), list):
        raise ValueError("Extraction payload must contain an ideas list")
    ideas = []
    for item in payload["ideas"]:
        if not isinstance(item, dict) or not text_value(item.get("idea_text")):
            raise ValueError("Each extracted idea needs nonempty idea_text")
        ideas.append({"idea_text": text_value(item["idea_text"]), "extraction_rationale": text_value(item.get("extraction_rationale"))})
    return ideas


# Enforce the allowed baseline statuses and taxonomy ID rules.
def validate_baseline(payload: dict | None, existing_ids: set[str]) -> dict:
    if not isinstance(payload, dict) or payload.get("classification_status") not in {"classified", "ambiguous", "no_fit"}:
        raise ValueError("Invalid baseline classification status")
    status = payload["classification_status"]
    ids = [str(x) for x in payload.get("policy_concept_ids", [])]
    if any(x not in existing_ids for x in ids):
        raise ValueError("Baseline classification referenced an unknown taxonomy ID")
    if status == "classified" and len(ids) != 1:
        raise ValueError("Classified baseline idea must have exactly one concept")
    if status == "no_fit" and ids:
        raise ValueError("No-fit baseline idea cannot have concepts")
    return {"classification_status": status, "policy_concept_ids": ids, "confidence": text_value(payload.get("confidence")).lower(), "rationale": text_value(payload.get("rationale"))}


# Format taxonomy rows as context for Cortex prompts.
def taxonomy_context(frame: pd.DataFrame) -> str:
    return "\n".join(
        f"ID={r.POLICY_CONCEPT_ID} | concept={r.POLICY_CONCEPT} | description={r.POLICY_CONCEPT_DESCRIPTION} | subtheme={r.SUBTHEME} | theme={r.THEME}"

        for r in frame.itertuples()
    )

In [5]:
# Build a normalized SQL input for each enabled survey field.
field_sql = {
    "government_action_suggestion": "government_action_suggestion",
    "economic_impact_expectation": "economic_impact_expectation",
}
field_queries = [
    f"SELECT top 20 survey_id, '{field}' AS source_field, {field_sql[field]} AS source_text FROM {SOURCE_SURVEY} WHERE NULLIF(TRIM({field_sql[field]}), '') IS NOT NULL"
    for field in EXTRACTION_FIELDS
]
survey_fields_sql = " UNION ALL ".join(field_queries)

# Run extraction for every response in one Snowflake SQL statement.
extraction_sql = f"""
WITH survey_fields AS (
    {survey_fields_sql}
)
SELECT
    survey_id,
    source_field,
    source_text,
    SNOWFLAKE.CORTEX.COMPLETE(
        %s,
        ARRAY_CONSTRUCT(
            OBJECT_CONSTRUCT('role', 'system', 'content', %s),
            OBJECT_CONSTRUCT('role', 'user', 'content', CONCAT('Source field: ', source_field, '\\nSurvey response:\\n', source_text))
        ),
        OBJECT_CONSTRUCT(
            'temperature', 0,
            'max_tokens', 1200,
            'response_format', PARSE_JSON(%s)
        )
    ) AS raw_cortex_response
FROM survey_fields
"""
extraction_response_schema = json.dumps({"type": "json", "schema": EXTRACTION_SCHEMA})
extraction_df = query_df(
    extraction_sql,
    [CORTEX_MODEL, EXTRACTION_PROMPT, extraction_response_schema],
)

# Validate and flatten each structured response while retaining malformed outputs for review.
def parse_extraction_row(row: pd.Series) -> list[dict]:
    base = {
        "run_id": RUN_ID,
        "run_timestamp": RUN_TIMESTAMP,
        "model_name": CORTEX_MODEL,
        "prompt_version": PROMPT_VERSION,
        "survey_id": row.SURVEY_ID,
        "source_field": row.SOURCE_FIELD,
        "source_text": row.SOURCE_TEXT,
        "source_fingerprint": stable_id(row.SURVEY_ID, row.SOURCE_FIELD, row.SOURCE_TEXT),
        "raw_cortex_response": row.RAW_CORTEX_RESPONSE,
    }
    try:
        raw = row.RAW_CORTEX_RESPONSE
        content = json.loads(extract_structured_response(raw))
        ideas = validate_extraction(content)
        if not ideas:
            return [{**base, "idea_id": stable_id(row.SURVEY_ID, row.SOURCE_FIELD, "NO_IDEA", row.SOURCE_TEXT), "idea_text": "", "extraction_rationale": "", "extraction_status": "no_ideas", "processing_error": None}]
        return [{**base, "idea_id": stable_id(row.SURVEY_ID, row.SOURCE_FIELD, idea["idea_text"]), **idea, "extraction_status": "extracted", "processing_error": None} for idea in ideas]
    except Exception as exc:
        return [{**base, "idea_id": stable_id(row.SURVEY_ID, row.SOURCE_FIELD, "INVALID", row.SOURCE_TEXT), "idea_text": "", "extraction_rationale": "", "extraction_status": "error", "processing_error": str(exc)}]


# Flatten the per-response JSON arrays into one visible dataframe of atomic ideas.
extraction_records = []
for row in extraction_df.itertuples(index=False):
    extraction_records.extend(parse_extraction_row(pd.Series(row._asdict())))
extraction_df = pd.DataFrame(extraction_records)
extraction_df.columns = [c.upper() for c in extraction_df.columns]
print(f"extraction inputs={len(extraction_df):,}; extracted result rows={len(extraction_df):,}")
extraction_df

extraction inputs=46; extracted result rows=46


,RUN_ID,RUN_TIMESTAMP,MODEL_NAME,PROMPT_VERSION,SURVEY_ID,SOURCE_FIELD,SOURCE_TEXT,SOURCE_FINGERPRINT,RAW_CORTEX_RESPONSE,IDEA_ID,IDEA_TEXT,EXTRACTION_RATIONALE,EXTRACTION_STATUS,PROCESSING_ERROR
0,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,c033a551-79e4-48f8-b7e8-dd13a4d74614,government_action_suggestion,"Oversight for safety, but don't let it become ...",6c2eb6192c42f227f7021dccfbb5a1bf,"{\n ""created"": 1788496359,\n ""model"": ""opena...",b79406c1cbc5f615eb28587dd907d6ce,Oversight for safety,The respondent explicitly recommends implement...,extracted,None
1,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,c033a551-79e4-48f8-b7e8-dd13a4d74614,government_action_suggestion,"Oversight for safety, but don't let it become ...",6c2eb6192c42f227f7021dccfbb5a1bf,"{\n ""created"": 1788496359,\n ""model"": ""opena...",7318b5111653595ca063d4e248d2a938,Don't let oversight become a slippery slope to...,The respondent explicitly advises against over...,extracted,None
2,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,7fcf5792-09d9-4735-bb25-2517a4e40980,government_action_suggestion,"I think the government should reign in AI, cre...",2f4b481f6e7387549b0091b993a9ad28,"{\n ""created"": 1788496363,\n ""model"": ""opena...",73709677f909ff1201c5e0c3fa8b360e,Reign in AI by creating common sense safety le...,Respondent explicitly advocates that the gover...,extracted,None
3,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,7fcf5792-09d9-4735-bb25-2517a4e40980,government_action_suggestion,"I think the government should reign in AI, cre...",2f4b481f6e7387549b0091b993a9ad28,"{\n ""created"": 1788496363,\n ""model"": ""opena...",d3cce369d303d29c57af76dacc33699d,Tax companies that are laying off workers beca...,Respondent explicitly advocates taxing compani...,extracted,None
4,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,7fcf5792-09d9-4735-bb25-2517a4e40980,government_action_suggestion,"I think the government should reign in AI, cre...",2f4b481f6e7387549b0091b993a9ad28,"{\n ""created"": 1788496363,\n ""model"": ""opena...",bfe0aad0c693998c86979ac24dc2a57c,Re-invest tax money into social benefit progra...,Respondent explicitly recommends reinvesting t...,extracted,None
5,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,34f4c4f7-a104-4fcd-8433-d689c11846d1,government_action_suggestion,They should fix themselves first instead of tr...,36d14bf62f7096dbfe13f169ff75f2e4,"{\n ""created"": 1788496360,\n ""model"": ""opena...",ffa9cdffeee5bc022fc20244bae1f379,They should fix themselves first instead of tr...,The respondent explicitly advises that 'They s...,extracted,None
6,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,34f4c4f7-a104-4fcd-8433-d689c11846d1,government_action_suggestion,They should fix themselves first instead of tr...,36d14bf62f7096dbfe13f169ff75f2e4,"{\n ""created"": 1788496360,\n ""model"": ""opena...",79916a64f1f08c2d057852fe74b21e57,Early government involvement and collaboration...,The respondent explicitly recommends early gov...,extracted,None
7,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,75be846d-db36-4cc1-a854-3e979fd81c23,government_action_suggestion,Limit AI use since it harms peoples. Look at o...,d73925beee75a32f2e5804ea7b46e7c7,"{\n ""created"": 1788496363,\n ""model"": ""opena...",c5315a1b857b61f8a85dc4adad0bc233,Limit AI use,The respondent explicitly advocates limiting A...,extracted,None
8,d433ef6f-2

In [19]:
# Keep only successfully extracted ideas as inputs to baseline classification.
ideas_df = extraction_df[
    (extraction_df["EXTRACTION_STATUS"] == "extracted")
    & extraction_df["IDEA_TEXT"].fillna("").str.strip().ne("")
].copy()
existing_ids = set(taxonomy_df["POLICY_CONCEPT_ID"].astype(str))
taxonomy_text = taxonomy_context(taxonomy_df)

# Classify every extracted idea in one SQL statement, with one Cortex call per SQL row.
baseline_sql = f"""
WITH {values_cte(ideas_df, ['IDEA_ID', 'SURVEY_ID', 'SOURCE_FIELD', 'IDEA_TEXT'])}
SELECT
    IDEA_ID,
    SURVEY_ID,
    SOURCE_FIELD,
    IDEA_TEXT,
    SNOWFLAKE.CORTEX.COMPLETE(
        %s,
        ARRAY_CONSTRUCT(
            OBJECT_CONSTRUCT('role', 'system', 'content', CONCAT(%s, '\\n\\nExisting taxonomy:\\n', %s)),
            OBJECT_CONSTRUCT('role', 'user', 'content', IDEA_TEXT)
        ),
        OBJECT_CONSTRUCT(
            'temperature', 0,
            'max_tokens', 1200,
            'response_format', PARSE_JSON(%s)
        )
    ) AS RAW_CORTEX_RESPONSE
FROM input_rows
"""
baseline_response_schema = json.dumps({"type": "json", "schema": CLASSIFICATION_SCHEMA})
baseline_raw_df = query_df(
    baseline_sql,
    [CORTEX_MODEL, BASELINE_PROMPT, taxonomy_text, baseline_response_schema],
)

# Validate baseline responses and enforce the allowed status/ID rules.
def parse_baseline_row(row: pd.Series) -> dict:
    base = {
        "run_id": RUN_ID,
        "run_timestamp": RUN_TIMESTAMP,
        "model_name": CORTEX_MODEL,
        "prompt_version": PROMPT_VERSION,
        "idea_id": row.IDEA_ID,
        "survey_id": row.SURVEY_ID,
        "source_field": row.SOURCE_FIELD,
        "idea_text": row.IDEA_TEXT,
        "raw_cortex_response": row.RAW_CORTEX_RESPONSE,
    }
    try:
        raw = json.loads(row.RAW_CORTEX_RESPONSE)
        result = validate_baseline(json.loads(extract_structured_response(row.RAW_CORTEX_RESPONSE)), existing_ids)
        return {**base, "classification_status": result["classification_status"], "policy_concept_ids": json.dumps(result["policy_concept_ids"]), "confidence": result["confidence"], "rationale": result["rationale"], "processing_error": None}
    except Exception as exc:
        return {**base, "classification_status": "error", "policy_concept_ids": "[]", "confidence": "low", "rationale": str(exc), "processing_error": str(exc)}


# Produce the visible baseline classification dataframe.
baseline_df = pd.DataFrame([parse_baseline_row(pd.Series(row._asdict())) for row in baseline_raw_df.itertuples(index=False)])
baseline_df.columns = [c.upper() for c in baseline_df.columns]
print(f"baseline ideas classified: {len(baseline_df):,}")
baseline_df

baseline ideas classified: 45


,RUN_ID,RUN_TIMESTAMP,MODEL_NAME,PROMPT_VERSION,IDEA_ID,SURVEY_ID,SOURCE_FIELD,IDEA_TEXT,RAW_CORTEX_RESPONSE,CLASSIFICATION_STATUS,POLICY_CONCEPT_IDS,CONFIDENCE,RATIONALE,PROCESSING_ERROR
0,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,b79406c1cbc5f615eb28587dd907d6ce,c033a551-79e4-48f8-b7e8-dd13a4d74614,government_action_suggestion,Oversight for safety,"{\n ""created"": 1788497211,\n ""model"": ""opena...",ambiguous,"[""5d6f894202f86106da1001fd5aa1f6be"", ""e0d68083...",medium,The idea 'Oversight for safety' could mean gen...,None
1,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,b1c7c68702596300ba5efa1609eac027,75be846d-db36-4cc1-a854-3e979fd81c23,government_action_suggestion,Look at other countries for inspiration,"{\n ""created"": 1788497211,\n ""model"": ""opena...",no_fit,[],high,The idea 'Look at other countries for inspirat...,None
2,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,82df5c04fb82544f34c408af8d98fb83,17fee2ed-9e96-42bc-b7ec-a34c5771e5f1,government_action_suggestion,Government should put sensible guide rails thr...,"{\n ""created"": 1788497210,\n ""model"": ""opena...",classified,"[""6f82c2410274c297972687991f6b5b5f""]",high,The proposal is to legislate guide rails to pr...,None
3,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,a618b6314801bd9259e2f76cba919337,c2043a33-ae5d-4924-af84-ad0cea9b5883,government_action_suggestion,Create several positions staffed by people who...,"{\n ""created"": 1788497211,\n ""model"": ""opena...",classified,"[""01bb012239f2cad5066368a00450d2ad""]",high,The proposal calls for creating positions staf...,None
4,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,607de2a0ef1ae28ac4ae53e1382c4261,290e4a79-69d4-41ac-b9a8-f2ef004d12e5,government_action_suggestion,Ensure workers are part of a protected class w...,"{\n ""created"": 1788497201,\n ""model"": ""opena...",classified,"[""92e3c18fb967d64d6a17e71751728018""]",high,The proposal—ensuring workers are a protected ...,None
5,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,d394da5bddc9efa5211b2323d18fb563,cf5e56db-1942-414a-9a9c-573d38f6c6a7,government_action_suggestion,Training for folks.,"{\n ""created"": 1788497200,\n ""model"": ""opena...",classified,"[""275645b7b1dc939841838e492f292dfb""]",high,The idea ‘Training for folks’ clearly refers t...,None
6,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,7318b5111653595ca063d4e248d2a938,c033a551-79e4-48f8-b7e8-dd13a4d74614,government_action_suggestion,Don't let oversight become a slippery slope to...,"{\n ""created"": 1788497211,\n ""model"": ""opena...",no_fit,[],high,The idea is a general normative principle abou...,None
7,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,d3c1e05467fbbcb5acb1b00901d37c7c,75be846d-db36-4cc1-a854-3e979fd81c23,government_action_suggestion,Look at history for similar historical patterns,"{\n ""created"": 1788497202,\n ""model"": ""opena...",no_fit,[],high,The idea 'Look at history for similar historic...,None
8,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,78f24b533424ee6b40b2e5e6a17662c7,17fee2ed-9e96-42bc-b7ec-a34c5771e5f1,government_action_suggestion,Government should put sensible guide rails thr...,"{\n ""created"": 1788497202,\n ""model"": ""opena...",classified,"[""6f82c2410274c297972687991f6b5b5f""]",high,The idea is about legislating protections fo

In [20]:
# Identify baseline ambiguous and no-fit ideas as possible taxonomy gaps.
baseline_all = baseline_df.copy()
baseline_all["POLICY_CONCEPT_IDS"] = baseline_all["POLICY_CONCEPT_IDS"].apply(json.loads)
gap_ideas = ideas_df.merge(baseline_all[["IDEA_ID", "CLASSIFICATION_STATUS", "POLICY_CONCEPT_IDS", "RATIONALE"]], on="IDEA_ID", how="left")
gap_ideas = gap_ideas[gap_ideas.CLASSIFICATION_STATUS.isin(["ambiguous", "no_fit"])]

# Ask Cortex to suggest candidate concepts only for ambiguous or no-fit ideas.
gap_context = "\n".join(f"idea_id={r.IDEA_ID} | status={r.CLASSIFICATION_STATUS} | idea={r.IDEA_TEXT}" for r in gap_ideas.itertuples())
gap_result, gap_raw, gap_error = cortex_json(
    [{"role": "system", "content": GAP_PROMPT + "\n\nExisting taxonomy:\n" + taxonomy_context(taxonomy_df)},
     {"role": "user", "content": gap_context or "No ambiguous or no-fit ideas."}],
    GAP_SCHEMA,
)

# Convert valid candidate concepts into an in-memory review dataframe.
candidate_records = []
if gap_error or gap_result is None:
    candidate_records.append({"run_id": RUN_ID, "run_timestamp": RUN_TIMESTAMP, "model_name": CORTEX_MODEL, "prompt_version": PROMPT_VERSION, "concept_status": "error", "raw_cortex_response": gap_raw, "processing_error": gap_error or "empty Cortex response"})
else:
    for candidate in gap_result.get("candidates", []):
        name = text_value(candidate.get("candidate_name"))
        if not name:
            continue
        candidate_records.append({
            "run_id": RUN_ID, "run_timestamp": RUN_TIMESTAMP, "model_name": CORTEX_MODEL, "prompt_version": PROMPT_VERSION,
            "policy_concept_id": stable_id(name), "policy_concept": name,
            "policy_concept_description": text_value(candidate.get("definition")), "subtheme": "", "theme": "",
            "concept_status": "new", "inclusion_criteria": text_value(candidate.get("boundary")),
            "exclusion_criteria": "Exclude ideas adequately represented by the closest existing concepts.",
            "supporting_idea_ids": json.dumps(candidate.get("supporting_idea_ids", [])),
            "why_existing_insufficient": text_value(candidate.get("why_existing_insufficient")),
            "closest_existing_concepts": json.dumps(candidate.get("closest_existing_concepts", [])),
            "raw_cortex_response": gap_raw, "processing_error": None,
        })
candidates_df = pd.DataFrame(candidate_records)
print(f"gap inputs={len(gap_ideas):,}; candidate concepts={len(candidates_df):,}")
candidates_df

gap inputs=26; candidate concepts=4


,run_id,run_timestamp,model_name,prompt_version,policy_concept_id,policy_concept,policy_concept_description,subtheme,theme,concept_status,inclusion_criteria,exclusion_criteria,supporting_idea_ids,why_existing_insufficient,closest_existing_concepts,raw_cortex_response,processing_error
0,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,cba5806994ce8126baf7838accc29b83,Establish independent AI safety oversight boar...,"Create an independent, multidisciplinary AI Sa...",,,new,"Policy would create specific, actionable overs...",Exclude ideas adequately represented by the cl...,"[""b79406c1cbc5f615eb28587dd907d6ce"", ""73709677...",Existing concepts call for openness and indepe...,"[""Make AI policymaking more open to the public...","{\n ""created"": 1788497239,\n ""model"": ""opena...",None
1,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,bd74d50d471e9b79f9e2a2f8bfd0b5fb,Domain‑specific deployment bans and conditiona...,"Introduce explicit, domain‑specific prohibitio...",,,new,Policy instrument limiting or phasing use of s...,Exclude ideas adequately represented by the cl...,"[""c5315a1b857b61f8a85dc4adad0bc233"", ""25ce7f91...",There are concepts about domain rules and limi...,"[""Set rules for where AI can be used safely"", ...","{\n ""created"": 1788497239,\n ""model"": ""opena...",None
2,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,4fc734f8f465d581a222570e0cf7eff4,Mandated early government consultation and int...,Require developers of large-scale or sensitive...,,,new,"Policy requiring early, documented government‑...",Exclude ideas adequately represented by the cl...,"[""79916a64f1f08c2d057852fe74b21e57"", ""b79406c1...",While existing entries encourage public‑sector...,"[""Use AI to improve how government works"", ""Ma...","{\n ""created"": 1788497239,\n ""model"": ""opena...",None
3,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,ba58238b32f7ea33cc8b75762d676112,Safeguards against oversight‑driven suppressio...,Require that any safety or oversight regime in...,,,new,Policy to protect free speech and prevent cens...,Exclude ideas adequately represented by the cl...,"[""7318b5111653595ca063d4e248d2a938""]",Existing concepts encourage openness and label...,"[""Make AI policymaking more open to the public...","{\n ""created"": 1788497239,\n ""model"": ""opena...",None


In [23]:
# Combine unchanged existing concepts with proposed new concepts for validation.
candidate_taxonomy = taxonomy_df.copy()
if not candidates_df.empty and "concept_status" in candidates_df:
    new_taxonomy = candidates_df.rename(columns={
        "policy_concept_id": "POLICY_CONCEPT_ID", "policy_concept": "POLICY_CONCEPT",
        "policy_concept_description": "POLICY_CONCEPT_DESCRIPTION", "subtheme": "SUBTHEME",
        "theme": "THEME", "concept_status": "CONCEPT_STATUS", "inclusion_criteria": "INCLUSION_CRITERIA",
        "exclusion_criteria": "EXCLUSION_CRITERIA",
    })
    candidate_taxonomy = pd.concat([candidate_taxonomy, new_taxonomy[taxonomy_df.columns]], ignore_index=True)

# Ask Cortex to identify taxonomy overlaps, gaps, and usable decision boundaries.
validation_user = "\n\nCandidate taxonomy:\n" + taxonomy_context(candidate_taxonomy) + "\n\nAtomic ideas:\n" + "\n".join(f"{r.IDEA_ID}: {r.IDEA_TEXT}" for r in ideas_df.itertuples())
validation_result, validation_raw, validation_error = cortex_json(
    [{"role": "system", "content": VALIDATION_PROMPT}, {"role": "user", "content": validation_user}],
    VALIDATION_SCHEMA,
    max_tokens=4000,
)

# Normalize validation findings for display without writing an intermediate table.
validation_findings = validation_result.get("findings", []) if validation_result else [{"severity": "high", "finding_type": "other", "concept_ids": [], "detail": validation_error or "empty Cortex response", "decision_rule": ""}]
validation_rows = [{"run_id": RUN_ID, "run_timestamp": RUN_TIMESTAMP, "model_name": CORTEX_MODEL, "prompt_version": PROMPT_VERSION, "is_ready": bool(validation_result and validation_result.get("is_ready") and not validation_error), "severity": f.get("severity", "high"), "finding_type": f.get("finding_type", "other"), "concept_ids": json.dumps(f.get("concept_ids", [])), "detail": text_value(f.get("detail")), "decision_rule": text_value(f.get("decision_rule")), "raw_cortex_response": validation_raw, "processing_error": validation_error} for f in validation_findings]
validation_df = pd.DataFrame(validation_rows)

# Add run metadata to the in-memory taxonomy snapshot; existing concepts remain unchanged.
candidate_taxonomy.columns = [c.lower() for c in candidate_taxonomy.columns]
candidate_taxonomy["run_id"] = RUN_ID
candidate_taxonomy["run_timestamp"] = RUN_TIMESTAMP
candidate_taxonomy["model_name"] = CORTEX_MODEL
candidate_taxonomy["prompt_version"] = PROMPT_VERSION

print(f"taxonomy concepts={len(candidate_taxonomy):,}; validation findings={len(validation_df):,}")
print("Validation findings:")
validation_df


taxonomy concepts=31; validation findings=10
Validation findings:


,run_id,run_timestamp,model_name,prompt_version,is_ready,severity,finding_type,concept_ids,detail,decision_rule,raw_cortex_response,processing_error
0,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,True,high,overlap,"[""b1ee0fd9f39e8f0afe81538358ec34e8"", ""5d6f8942...",Material overlap between concepts covering dom...,Treat 'set rules for where AI can be used' and...,"{\n ""created"": 1788497765,\n ""model"": ""opena...",None
1,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,True,high,mixed_dimension,"[""35f14fb39159c2e6a5dffbee4d992b5a"", ""6f82c241...",Overlap and potential duplicate/adjacent cover...,Differentiate liability/IP/wealth distribution...,"{\n ""created"": 1788497765,\n ""model"": ""opena...",None
2,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,True,medium,overlap,"[""2358e7bd69341e13635d2af29546920d"", ""c3fc6e65...",Overlap between data-center environmental prot...,Treat environmental/community impact disclosur...,"{\n ""created"": 1788497765,\n ""model"": ""opena...",None
3,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,True,medium,ambiguity,"[""32327f00d78630831a714625eb862fc5"", ""f47a9e0e...",Overlap between general transparency about per...,Distinguish 'personal data handling/transparen...,"{\n ""created"": 1788497765,\n ""model"": ""opena...",None
4,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,True,medium,granularity,"[""155ace4884b019215cf74a50eca4e530"", ""d04c9304...",Good coverage but several atomic items combine...,Separate worker-centered policies into (a) pro...,"{\n ""created"": 1788497765,\n ""model"": ""opena...",None
5,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,True,low,ambiguity,"[""54d3e4734f305d71def1754f0f0550b1"", ""e0d68083...","Some atomic ideas (e.g., 'Use AI to solve prob...",Treat human-centered preservation concepts sep...,"{\n ""created"": 1788497765,\n ""model"": ""opena...",None
6,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,True,low,parent_child,"[""662d7315e2f91a4477592d211cdc90ae"", ""a7ae5a71...",Overlap between school protections and communi...,Education-related concepts: map K‑12 curricula...,"{\n ""created"": 1788497765,\n ""model"": ""opena...",None
7,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,True,medium,parent_child,"[""cba5806994ce8126baf7838accc29b83"", ""4fc734f8...",Overlap between establishing oversight bodies ...,Oversight and free-speech protections: treat a...,"{\n ""created"": 1788497765,\n ""model"": ""opena...",None
8,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,True,medium,overlap,"[""32327f00d78630831a714625eb862fc5"", ""cc70356e...","Some atomic items conflate IP protection, tran...",Intellectual property and data-use protection:...,"{\n ""created"": 1788497765,\n ""model"": ""opena...",None
9,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,True,low,granularity,"[""a3432ff3a4a87749973c978328656a06"", ""3fa831f9...",Atomic idea 'They should fix themselves first ...,Differentiate 'support independent research' (...,"{\n ""created"": 1788497765,\n ""model"": ""opena...",None


In [28]:
# Allow final classification only after review mode is disabled and taxonomy validation passes.
taxonomy_ready = bool(validation_result and validation_result.get("is_ready") and not validation_error)

if REVIEW_MODE or not taxonomy_ready:
    final_df = pd.DataFrame()
    reason = "REVIEW_MODE=True" if REVIEW_MODE else "taxonomy validation did not report ready"
    print(f"Final classification skipped because {reason}.")
else:
    # Use the validated in-memory taxonomy as the only allowed classification vocabulary.
    frozen_taxonomy = candidate_taxonomy.copy()
    frozen_ids = set(frozen_taxonomy["policy_concept_id"])
    frozen_context = taxonomy_context(frozen_taxonomy.rename(columns=str.upper))

    # Classify every extracted idea in one SQL statement, with one Cortex call per SQL row.
    final_sql = f"""
    WITH {values_cte(ideas_df, ['IDEA_ID', 'SURVEY_ID', 'SOURCE_FIELD', 'IDEA_TEXT'])}
    SELECT
        IDEA_ID,
        SURVEY_ID,
        SOURCE_FIELD,
        IDEA_TEXT,
        SNOWFLAKE.CORTEX.COMPLETE(
            %s,
            ARRAY_CONSTRUCT(
                OBJECT_CONSTRUCT('role', 'system', 'content', CONCAT(%s, '\\n\\nFrozen taxonomy:\\n', %s)),
                OBJECT_CONSTRUCT('role', 'user', 'content', IDEA_TEXT)
            ),
            OBJECT_CONSTRUCT(
                'temperature', 0,
                'max_tokens', 1200,
                'response_format', PARSE_JSON(%s)
            )
        ) AS RAW_CORTEX_RESPONSE
    FROM input_rows
    """
    final_response_schema = json.dumps({"type": "json", "schema": FINAL_SCHEMA})
    final_raw_df = query_df(
        final_sql,
        [CORTEX_MODEL, FINAL_PROMPT, frozen_context, final_response_schema],
    )

    # Validate final responses and enforce at most one known concept per idea.
    def parse_final_row(row: pd.Series) -> dict:
        base = {
            "run_id": RUN_ID,
            "run_timestamp": RUN_TIMESTAMP,
            "model_name": CORTEX_MODEL,
            "prompt_version": PROMPT_VERSION,
            "survey_id": row.SURVEY_ID,
            "source_field": row.SOURCE_FIELD,
            "idea_id": row.IDEA_ID,
            "idea_text": row.IDEA_TEXT,
            "raw_cortex_response": row.RAW_CORTEX_RESPONSE,
        }
        try:
            result = json.loads(extract_structured_response(row.RAW_CORTEX_RESPONSE))
            status = result.get("classification_status")
            concept_id = result.get("policy_concept_id")
            if status not in {"classified", "ambiguous", "no_fit"}:
                raise ValueError("Invalid final classification status")
            if status == "classified" and concept_id not in frozen_ids:
                raise ValueError("Final classifier returned an unknown concept ID")
            if status != "classified" and concept_id is not None:
                raise ValueError("Only classified ideas may contain a concept ID")
            concept = frozen_taxonomy.loc[frozen_taxonomy.policy_concept_id == concept_id, "policy_concept"].iloc[0] if concept_id else None
            return {**base, "classification_status": status, "policy_concept_id": concept_id, "policy_concept": concept, "classification_confidence": text_value(result.get("confidence")).lower(), "classification_rationale": text_value(result.get("rationale")), "processing_error": None}
        except Exception as exc:
            return {**base, "classification_status": "error", "policy_concept_id": None, "policy_concept": None, "classification_confidence": "low", "classification_rationale": str(exc), "processing_error": str(exc)}

    # Produce the visible final classification dataframe.
    final_df = pd.DataFrame([parse_final_row(pd.Series(row._asdict())) for row in final_raw_df.itertuples(index=False)])
    final_df.columns = [c.upper() for c in final_df.columns]
    print(f"final ideas classified: {len(final_df):,}")
    final_df

final ideas classified: 45


In [29]:
# Use the in-memory results to build review queues without reading or writing output tables.
all_extractions = extraction_df.copy()
all_final = final_df.copy()

# Identify responses and ideas requiring manual review.
response_ids_with_ideas = set(all_extractions.loc[all_extractions.EXTRACTION_STATUS == "extracted", "SURVEY_ID"])
no_idea_responses = survey_df[~survey_df.SURVEY_ID.isin(response_ids_with_ideas)].copy()
ambiguous = all_final[all_final.CLASSIFICATION_STATUS == "ambiguous"].copy() if not all_final.empty else pd.DataFrame()
no_fit = all_final[all_final.CLASSIFICATION_STATUS == "no_fit"].copy() if not all_final.empty else pd.DataFrame()
low_confidence = all_final[all_final.CLASSIFICATION_CONFIDENCE == "low"].copy() if not all_final.empty else pd.DataFrame()
errors = all_extractions[all_extractions.EXTRACTION_STATUS == "error"].copy()
if not all_final.empty and "PROCESSING_ERROR" in all_final:
    errors = pd.concat([errors, all_final[all_final.PROCESSING_ERROR.notna()]], ignore_index=True)

# Summarize the size of each review queue.
qa_summary = pd.DataFrame([
    {"review_category": "responses_without_ideas", "count": len(no_idea_responses)},
    {"review_category": "ambiguous_final_classifications", "count": len(ambiguous)},
    {"review_category": "no_fit_final_classifications", "count": len(no_fit)},
    {"review_category": "low_confidence_classifications", "count": len(low_confidence)},
    {"review_category": "errors", "count": len(errors)},
])
qa_summary

,review_category,count
0,responses_without_ideas,2
1,ambiguous_final_classifications,17
2,no_fit_final_classifications,8
3,low_confidence_classifications,1
4,errors,0


In [30]:
# Display counts by final status and concept; only classified ideas contribute to concept counts.
if not all_final.empty:
    classified = all_final[all_final.CLASSIFICATION_STATUS == "classified"]
    status_counts = all_final["CLASSIFICATION_STATUS"].value_counts(dropna=False).rename_axis("classification_status").reset_index(name="count")
    concept_counts = classified.groupby(["POLICY_CONCEPT_ID", "POLICY_CONCEPT"]).size().reset_index(name="idea_count").sort_values("idea_count", ascending=False)
else:
    status_counts = pd.DataFrame(columns=["classification_status", "count"])
    concept_counts = pd.DataFrame(columns=["POLICY_CONCEPT_ID", "POLICY_CONCEPT", "idea_count"])

print("Status counts:")
status_counts

Status counts:


,classification_status,count
0,classified,20
1,ambiguous,17
2,no_fit,8


In [31]:
# Display the complete existing-plus-new taxonomy for review.
candidate_taxonomy

,policy_concept_id,policy_concept,policy_concept_description,subtheme,theme,concept_status,inclusion_criteria,exclusion_criteria,run_id,run_timestamp,model_name,prompt_version
0,35f14fb39159c2e6a5dffbee4d992b5a,Hold AI companies liable for harm they cause,Hold the companies that build and deploy AI li...,Protecting people and nature from the effects ...,Making rules for AI companies,existing,,,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local
1,b1ee0fd9f39e8f0afe81538358ec34e8,Set rules for where AI can be used safely,Set domain-specific rules that keep human judg...,Regulating how AI works and where it’s used,Making rules for AI companies,existing,,,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local
2,2358e7bd69341e13635d2af29546920d,Protect people and nature from the resource ne...,Regulate the environmental and community impac...,Protecting people and nature from the effects ...,Making rules for AI companies,existing,,,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local
3,32327f00d78630831a714625eb862fc5,Make companies follow rules and be open about ...,Require companies to be transparent about how ...,Empowering people to protect their data,Protecting our way of life,existing,,,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local
4,155ace4884b019215cf74a50eca4e530,Make companies reskill or give career help to ...,Fund reskilling and career-transition pathways...,Protecting jobs and workers,Protecting our way of life,existing,,,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local
5,5d6f894202f86106da1001fd5aa1f6be,Set rules around how AI works,Require transparency into how AI systems opera...,Regulating how AI works and where it’s used,Making rules for AI companies,existing,,,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local
6,d04c9304854c075fa466ad6689bee198,"Deter layoffs because of AI, reward firms that...",Reward firms for retaining and hiring workers ...,Protecting jobs and workers,Protecting our way of life,existing,,,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local
7,01bb012239f2cad5066368a00450d2ad,Make AI policymaking more open to the public,Keep AI policymaking open to the public and re...,Making sure the public has their say on AI imp...,Protecting our way of life,existing,,,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local
8,54d3e4734f305d71def1754f0f0550b1,Set rules for labeling when AI is used,Require clear disclosure whenever people are d...,Preserving the rights of humans to choose and ...,Protecting our way of life,existing,,,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local
9,a0d0ba66a76b69d970a309fb28716c9d,Restrict and tax AI companies’ profits,Address the concentration of AI's economic gai...,Regulating how AI works and where it’s used,Making rules for AI companies,existing,,,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local


In [32]:
# Display taxonomy validation findings and any Cortex validation error.
validation_df

,run_id,run_timestamp,model_name,prompt_version,is_ready,severity,finding_type,concept_ids,detail,decision_rule,raw_cortex_response,processing_error
0,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,True,high,overlap,"[""b1ee0fd9f39e8f0afe81538358ec34e8"", ""5d6f8942...",Material overlap between concepts covering dom...,Treat 'set rules for where AI can be used' and...,"{\n ""created"": 1788497765,\n ""model"": ""opena...",None
1,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,True,high,mixed_dimension,"[""35f14fb39159c2e6a5dffbee4d992b5a"", ""6f82c241...",Overlap and potential duplicate/adjacent cover...,Differentiate liability/IP/wealth distribution...,"{\n ""created"": 1788497765,\n ""model"": ""opena...",None
2,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,True,medium,overlap,"[""2358e7bd69341e13635d2af29546920d"", ""c3fc6e65...",Overlap between data-center environmental prot...,Treat environmental/community impact disclosur...,"{\n ""created"": 1788497765,\n ""model"": ""opena...",None
3,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,True,medium,ambiguity,"[""32327f00d78630831a714625eb862fc5"", ""f47a9e0e...",Overlap between general transparency about per...,Distinguish 'personal data handling/transparen...,"{\n ""created"": 1788497765,\n ""model"": ""opena...",None
4,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,True,medium,granularity,"[""155ace4884b019215cf74a50eca4e530"", ""d04c9304...",Good coverage but several atomic items combine...,Separate worker-centered policies into (a) pro...,"{\n ""created"": 1788497765,\n ""model"": ""opena...",None
5,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,True,low,ambiguity,"[""54d3e4734f305d71def1754f0f0550b1"", ""e0d68083...","Some atomic ideas (e.g., 'Use AI to solve prob...",Treat human-centered preservation concepts sep...,"{\n ""created"": 1788497765,\n ""model"": ""opena...",None
6,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,True,low,parent_child,"[""662d7315e2f91a4477592d211cdc90ae"", ""a7ae5a71...",Overlap between school protections and communi...,Education-related concepts: map K‑12 curricula...,"{\n ""created"": 1788497765,\n ""model"": ""opena...",None
7,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,True,medium,parent_child,"[""cba5806994ce8126baf7838accc29b83"", ""4fc734f8...",Overlap between establishing oversight bodies ...,Oversight and free-speech protections: treat a...,"{\n ""created"": 1788497765,\n ""model"": ""opena...",None
8,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,True,medium,overlap,"[""32327f00d78630831a714625eb862fc5"", ""cc70356e...","Some atomic items conflate IP protection, tran...",Intellectual property and data-use protection:...,"{\n ""created"": 1788497765,\n ""model"": ""opena...",None
9,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,True,low,granularity,"[""a3432ff3a4a87749973c978328656a06"", ""3fa831f9...",Atomic idea 'They should fix themselves first ...,Differentiate 'support independent research' (...,"{\n ""created"": 1788497765,\n ""model"": ""opena...",None


In [33]:
# Display responses for which Cortex extracted no policy ideas.
no_idea_responses

,SURVEY_ID,GOVERNMENT_ACTION_SUGGESTION,ECONOMIC_IMPACT_EXPECTATION
15,e005ebad-6945-4900-aaf0-a6cd6eca9543,Large Language Models (LLMs) with Machine Lear...,Large Language Models (LLMs) with Machine Lear...
17,8e3671d6-870c-4a67-88c2-5630f31e3d07,None,AI has already impacted the economy. From hard...


In [34]:
# Display ambiguous final classifications for manual review.
ambiguous

,RUN_ID,RUN_TIMESTAMP,MODEL_NAME,PROMPT_VERSION,SURVEY_ID,SOURCE_FIELD,IDEA_ID,IDEA_TEXT,RAW_CORTEX_RESPONSE,CLASSIFICATION_STATUS,POLICY_CONCEPT_ID,POLICY_CONCEPT,CLASSIFICATION_CONFIDENCE,CLASSIFICATION_RATIONALE,PROCESSING_ERROR
0,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,c033a551-79e4-48f8-b7e8-dd13a4d74614,government_action_suggestion,b79406c1cbc5f615eb28587dd907d6ce,Oversight for safety,"{\n ""created"": 1788497983,\n ""model"": ""opena...",ambiguous,None,None,medium,The short phrase 'Oversight for safety' is amb...,None
1,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,75be846d-db36-4cc1-a854-3e979fd81c23,government_action_suggestion,b1c7c68702596300ba5efa1609eac027,Look at other countries for inspiration,"{\n ""created"": 1788497983,\n ""model"": ""opena...",ambiguous,None,None,high,The idea 'Look at other countries for inspirat...,None
4,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,290e4a79-69d4-41ac-b9a8-f2ef004d12e5,government_action_suggestion,607de2a0ef1ae28ac4ae53e1382c4261,Ensure workers are part of a protected class w...,"{\n ""created"": 1788497983,\n ""model"": ""opena...",ambiguous,None,None,high,The proposal 'Ensure workers are part of a pro...,None
5,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,cf5e56db-1942-414a-9a9c-573d38f6c6a7,government_action_suggestion,d394da5bddc9efa5211b2323d18fb563,Training for folks.,"{\n ""created"": 1788497991,\n ""model"": ""opena...",ambiguous,None,None,high,The input 'Training for folks.' is too short a...,None
9,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,dd80c22f-9f86-494a-8fa6-80de036d7673,government_action_suggestion,25ce7f9153f5d40128cfa6f69d95c9d9,Curb AI usage immediately,"{\n ""created"": 1788497992,\n ""model"": ""opena...",ambiguous,None,None,high,The request 'Curb AI usage immediately' is a h...,None
10,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,290e4a79-69d4-41ac-b9a8-f2ef004d12e5,government_action_suggestion,b1032ec80bed4d56f610321e0552245d,Use AI to solve problems and train AI to have ...,"{\n ""created"": 1788497980,\n ""model"": ""opena...",ambiguous,None,None,medium,The idea combines two distinct proposals: usin...,None
12,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,7fcf5792-09d9-4735-bb25-2517a4e40980,government_action_suggestion,73709677f909ff1201c5e0c3fa8b360e,Reign in AI by creating common sense safety le...,"{\n ""created"": 1788497992,\n ""model"": ""opena...",ambiguous,None,None,medium,The suggestion “Reign in AI by creating common...,None
19,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,9b185225-611f-426a-81de-72a35ec4fe22,government_action_suggestion,3ec343cdf41940912b81a0182b831852,AI should be heavily regulated.,"{\n ""created"": 1788497983,\n ""model"": ""opena...",ambiguous,None,None,low,The input statement 'AI should be heavily regu...,None
21,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,50a4cff1-1e78-4628-80b7-f8098bf92348,government_action_suggestion,fe65cfae365c18fab4602d7806350c3e,Limit AI use,"{\n ""created"": 1788497979,\n ""model"": ""opena...",ambiguous,None,None,high,The short phrase 'Limit AI use' is underspecif...,None
26,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,e22fcea4-7947-45f2-890b-71118fba2e01,government_action_sugge

In [35]:
# Display final no-fit classifications for manual review.
no_fit

,RUN_ID,RUN_TIMESTAMP,MODEL_NAME,PROMPT_VERSION,SURVEY_ID,SOURCE_FIELD,IDEA_ID,IDEA_TEXT,RAW_CORTEX_RESPONSE,CLASSIFICATION_STATUS,POLICY_CONCEPT_ID,POLICY_CONCEPT,CLASSIFICATION_CONFIDENCE,CLASSIFICATION_RATIONALE,PROCESSING_ERROR
7,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,75be846d-db36-4cc1-a854-3e979fd81c23,government_action_suggestion,d3c1e05467fbbcb5acb1b00901d37c7c,Look at history for similar historical patterns,"{\n ""created"": 1788497982,\n ""model"": ""opena...",no_fit,None,None,high,"The request ""Look at history for similar histo...",None
11,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,cf5e56db-1942-414a-9a9c-573d38f6c6a7,government_action_suggestion,82a27ce5065852955bb17174a3b8e7d6,Access to AI tools; roll out MS Copilot to all...,"{\n ""created"": 1788497980,\n ""model"": ""opena...",no_fit,None,None,high,The idea is an organizational decision to roll...,None
22,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,290e4a79-69d4-41ac-b9a8-f2ef004d12e5,government_action_suggestion,37f699afd5b604b4f49546743f6860f9,Build new societal foundations (build back bet...,"{\n ""created"": 1788497991,\n ""model"": ""opena...",no_fit,None,None,high,"The statement is a broad value-driven goal—""Bu...",None
30,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,34f4c4f7-a104-4fcd-8433-d689c11846d1,government_action_suggestion,ffa9cdffeee5bc022fc20244bae1f379,They should fix themselves first instead of tr...,"{\n ""created"": 1788497982,\n ""model"": ""opena...",no_fit,None,None,high,The statement is an opinion about regulatory t...,None
33,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,290e4a79-69d4-41ac-b9a8-f2ef004d12e5,government_action_suggestion,5417e4a316cb78a011e83080ff331ce1,Ensure that citizens are well educated to part...,"{\n ""created"": 1788497980,\n ""model"": ""opena...",no_fit,None,None,high,The policy idea—ensuring citizens are well edu...,None
39,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,5ad59128-47c2-41c8-9843-6b64fff639d2,government_action_suggestion,50f7bb85606dea92db40ccfcfc3e3eb8,Ban AI use for profit and education.,"{\n ""created"": 1788497991,\n ""model"": ""opena...",no_fit,None,None,high,The proposal 'Ban AI use for profit and educat...,None
41,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,45e3dc2f-3a41-4699-9c14-3896ddb2fa6a,government_action_suggestion,15355f451cc2d282d67b37face2889cc,AI must be seen as a product of all humanity.,"{\n ""created"": 1788497983,\n ""model"": ""opena...",no_fit,None,None,high,The statement 'AI must be seen as a product of...,None
43,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,290e4a79-69d4-41ac-b9a8-f2ef004d12e5,government_action_suggestion,3218cd97706f123ded130cb95afa0005,"Ensure everyone has access to housing, food, c...","{\n ""created"": 1788497983,\n ""model"": ""opena...",no_fit,None,None,high,The proposed idea is a broad social rights/gua...,None


In [36]:
# Display final classifications with low confidence.
low_confidence

,RUN_ID,RUN_TIMESTAMP,MODEL_NAME,PROMPT_VERSION,SURVEY_ID,SOURCE_FIELD,IDEA_ID,IDEA_TEXT,RAW_CORTEX_RESPONSE,CLASSIFICATION_STATUS,POLICY_CONCEPT_ID,POLICY_CONCEPT,CLASSIFICATION_CONFIDENCE,CLASSIFICATION_RATIONALE,PROCESSING_ERROR
19,d433ef6f-2b30-47f1-a3e9-c03c40618128,2026-09-04T04:31:25.663171+00:00,openai-gpt-5-mini,policy-concepts-v3-set-based-sql-local,9b185225-611f-426a-81de-72a35ec4fe22,government_action_suggestion,3ec343cdf41940912b81a0182b831852,AI should be heavily regulated.,"{\n ""created"": 1788497983,\n ""model"": ""opena...",ambiguous,None,None,low,The input statement 'AI should be heavily regu...,None


In [37]:
# Display malformed or failed Cortex results for debugging.
errors

,RUN_ID,RUN_TIMESTAMP,MODEL_NAME,PROMPT_VERSION,SURVEY_ID,SOURCE_FIELD,SOURCE_TEXT,SOURCE_FINGERPRINT,RAW_CORTEX_RESPONSE,IDEA_ID,IDEA_TEXT,EXTRACTION_RATIONALE,EXTRACTION_STATUS,PROCESSING_ERROR,CLASSIFICATION_STATUS,POLICY_CONCEPT_ID,POLICY_CONCEPT,CLASSIFICATION_CONFIDENCE,CLASSIFICATION_RATIONALE


In [38]:
# Write the frozen taxonomy only when explicitly enabled.
if ENABLE_OUTPUT_WRITES:
    write_pandas(
        conn,
        candidate_taxonomy,
        OPTIONAL_TAXONOMY_TABLE,
        database=TARGET_DATABASE,
        schema=TARGET_SCHEMA,
        auto_create_table=True,
        overwrite=True,
        quote_identifiers=True,
    )
    print(f"Wrote {len(candidate_taxonomy):,} taxonomy rows to {TARGET_DATABASE}.{TARGET_SCHEMA}.{OPTIONAL_TAXONOMY_TABLE}.")
else:
    print("Taxonomy write skipped; set ENABLE_OUTPUT_WRITES = True to opt in.")

Taxonomy write skipped; set ENABLE_OUTPUT_WRITES = True to opt in.


In [39]:
# Write final classifications only when explicitly enabled.
if ENABLE_OUTPUT_WRITES and not final_df.empty:
    write_pandas(
        conn,
        final_df,
        OPTIONAL_CLASSIFICATION_TABLE,
        database=TARGET_DATABASE,
        schema=TARGET_SCHEMA,
        auto_create_table=True,
        overwrite=True,
        quote_identifiers=True,
    )
    print(f"Wrote {len(final_df):,} classification rows to {TARGET_DATABASE}.{TARGET_SCHEMA}.{OPTIONAL_CLASSIFICATION_TABLE}.")
else:
    print("Classification write skipped; set ENABLE_OUTPUT_WRITES = True after review to opt in.")

Classification write skipped; set ENABLE_OUTPUT_WRITES = True after review to opt in.


In [40]:
# Close the local cursor and connection after all desired cells have been inspected.
cur.close()
conn.close()
print("Snowflake connection closed.")

Snowflake connection closed.
